# Phishing Website Detection — Step 4: From Dataset Features to Real-Time Features

**Capstone roadmap:**
1. ~~Dataset acquisition & exploration~~ ✅
2. ~~Baseline model + evaluation~~ ✅
3. ~~Random Forest, tuning, saved model~~ ✅
4. **Step 4 (this notebook): make the model deployable — retrain on live-extractable features**
5. Wrap it in an API + browser extension

### The problem

Your Step 3 model was trained on 30 features that were **pre-extracted** by the dataset's creators — several of them (domain age, Google PageRank, Alexa traffic rank) rely on services that are slow, paid, or simply don't exist anymore. A browser extension needs an instant answer from just a URL string.

So this notebook retrains the model on a **smaller, honest feature set** — one that `feature_extractor.py` (the companion file) can actually compute live, from a URL alone, with zero network calls. We're trading some raw accuracy for something that actually works in production. That trade-off is normal engineering, and worth stating explicitly rather than hiding it.

In [ ]:
import pandas as pd
import joblib
import sys
sys.path.append('../src')  # so we can import feature_extractor.py from notebooks/

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, classification_report
)

from feature_extractor import FEATURE_NAMES, extract_features_dict

**Before running the next cell:** make sure `feature_extractor.py` is saved inside your `src/` folder (not `notebooks/`). If you haven't moved it there yet, do that now.

## 1. Which of our 30 original columns match the live-extractable features?

`FEATURE_NAMES` (imported above) lists exactly what `feature_extractor.py` can compute from a URL string alone. Most of these names match column names already in our dataset — except `uses_https`, which is a new, weaker approximation of the original `SSLfinal_State` column (that one required a real certificate check over the network).

In [ ]:
df = pd.read_csv('phishing_data.csv')

print("Live-extractable feature names:", FEATURE_NAMES)
print("\nWhich of these already exist as columns in our dataset?")
for name in FEATURE_NAMES:
    print(f"  {name}: {'found' if name in df.columns else 'NOT in dataset'}")

`uses_https` won't be found — that's expected, since it's a new feature we invented as a stand-in for `SSLfinal_State`. We'll approximate it using the existing `SSLfinal_State` column for training purposes (it's not a perfect match, but it's the closest available proxy in this dataset — another documented simplification).

In [ ]:
# Build our reduced training set using the matching columns,
# and use SSLfinal_State as the training stand-in for uses_https.
reduced_columns = [c for c in FEATURE_NAMES if c != 'uses_https']

X_reduced = df[reduced_columns].copy()
X_reduced['uses_https'] = df['SSLfinal_State']

# Reorder columns to exactly match FEATURE_NAMES — this order must be identical
# at both training time and prediction time, or the model will misread inputs.
X_reduced = X_reduced[FEATURE_NAMES]

y = df['Result']

X_reduced.head()

## 2. Split and retrain on this reduced feature set

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_reduced, y, test_size=0.2, stratify=y, random_state=42
)

live_model = RandomForestClassifier(n_estimators=200, random_state=42)
live_model.fit(X_train, y_train)

y_pred = live_model.predict(X_test)

## 3. Evaluate — and be honest about the trade-off

Compare this against your Step 3 results (30 features). We expect this to be somewhat **lower**, since we've deliberately dropped 20 features, many of which were quite informative (HTML/JS-based and domain-based signals). That drop is the price of being deployable in real time.

In [ ]:
print("=== Reduced (live-extractable) Random Forest ===")
print("Accuracy: ", round(accuracy_score(y_test, y_pred), 3))
print("Precision (phishing):", round(precision_score(y_test, y_pred, pos_label=-1), 3))
print("Recall (phishing):   ", round(recall_score(y_test, y_pred, pos_label=-1), 3))
print("F1 (phishing):        ", round(f1_score(y_test, y_pred, pos_label=-1), 3))
print()
print(classification_report(y_test, y_pred, target_names=['Phishing (-1)', 'Legitimate (1)']))

**Your task:** fill in your Step 3 numbers here for side-by-side comparison, then note how big the gap is. A small gap means the trade-off was cheap; a large gap is worth mentioning explicitly in your capstone report as a known limitation and a direction for future work (e.g. optionally fetching page HTML for a slower-but-more-accurate 'deep scan' mode).

- Step 3 (30 features) F1: ___
- Step 4 (10 features) F1: ___

## 4. Sanity-check: run the real extractor on real URLs, then predict

This is the actual end-to-end path our future API will use: raw URL → `extract_features()` → model → prediction. Let's prove it works before we build anything else on top of it.

In [ ]:
test_urls = [
    "https://www.google.com",
    "https://www.wikipedia.org",
    "http://192.168.1.1/login/verify",
    "http://paypal-secure-login.tk/account@confirm",
]

for url in test_urls:
    features_dict = extract_features_dict(url)
    features_row = pd.DataFrame([features_dict])[FEATURE_NAMES]  # correct column order
    prediction = live_model.predict(features_row)[0]
    label = "PHISHING" if prediction == -1 else "legitimate"
    print(f"{url}\n  -> predicted: {label}\n")

Check these predictions make intuitive sense. If something looks wrong, it's worth digging into *why* — that's a more valuable debugging exercise than it might seem, and exactly the kind of thing your capstone report/viva may ask about.

## 5. Save the deployable model

This is the file the API in the next step will load. We keep it separate from Step 3's `phishing_model.pkl` (which used the full 30-feature set and isn't deployable as-is) so both remain on record — useful for your report, showing the accuracy-vs-deployability trade-off explicitly.

In [ ]:
import os
os.makedirs('../models', exist_ok=True)

joblib.dump(live_model, '../models/phishing_model_live.pkl')
joblib.dump(FEATURE_NAMES, '../models/feature_columns_live.pkl')

print("Saved models/phishing_model_live.pkl")
print("Saved models/feature_columns_live.pkl")

## Summary — what we now have

- [ ] A clear understanding of *why* a dataset feature and a live-extractable feature aren't always the same thing
- [ ] A working `feature_extractor.py` that turns a raw URL into a feature vector, with zero network calls
- [ ] A model retrained specifically to match that feature set (so training and prediction-time inputs are consistent)
- [ ] A validated end-to-end path: URL → features → prediction
- [ ] `models/phishing_model_live.pkl` — the file our API will load

**Next up (Step 5):** wrap `phishing_model_live.pkl` and `feature_extractor.py` in a small Flask API with one endpoint: send it a URL, get back a phishing/legitimate verdict.